# 01_setup_and_data_check

## Dataset sizes per split

Counts the manifest rows for train, val, and test for each task (`TASKS` in `woofdata.tasks`) and each dataset (source) the task uses.

Two tables:
- **Per task and source**: one row for each dataset a task uses. A dataset used by two tasks (PlayPanT) shows up once per task.
- **Per task**: the same counts added up for each task.

Train counts both `train` and `train_only` rows, which is what `task_rows(task, "train")` returns. `train_only` rows are used for training but never for scoring.

In [ ]:
import pandas as pd

from woofdata.tasks import EVAL_SPLITS, TASKS, read_manifest

# ["train", "val", "test"]
SPLIT_COLUMNS = ["train", *EVAL_SPLITS]
manifest = read_manifest()

size_rows = []
for task_name, task in TASKS.items():
    for source in task.sources:
        # for each source, count the splits
        source_rows = manifest.loc[manifest["source"] == source]
        source_splits = source_rows["split"]

        # count the number of rows for each split
        counts = {"task": task_name, "source": source}

        # "train" counts train + train_only, as task_rows(task, "train") does.
        counts["train"] = source_splits.isin(task.train_splits).sum()

        # count the number of rows for each eval split
        for split in EVAL_SPLITS:
            counts[split] = (source_splits == split).sum()
        
        size_rows.append(counts)

sizes = pd.DataFrame(size_rows)
sizes["total"] = sizes[SPLIT_COLUMNS].sum(axis=1)

print("Per task and source:")
display(sizes)
print("Per task:")
display(sizes.groupby("task", sort=False)[[*SPLIT_COLUMNS, "total"]].sum())

## Posture labels: look at the images

For each posture shown (`lying`, then `standing`), picks 20 random DogPoseCV train images with that label and draws the dog's box in red. Use these to check by eye that the labels and boxes are right before training the posture model.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import patches

from woofdata.loader import load_sample
from woofdata.tasks import task_rows

NUM_IMAGES = 20
rows = task_rows("posture", "train")

def show_posture(posture: str) -> None:
    # posture_rows is a DataFrame containing all rows with the given posture label
    posture_rows = rows[rows["original_label"] == posture]
    # randomly sample NUM_IMAGES rows from posture_rows
    picked_rows = posture_rows.sample(NUM_IMAGES, random_state=0)

    # Draw the images in a grid with 4 rows and 5 columns
    figure, axes = plt.subplots(4, 5, figsize=(15, 12))
    figure.suptitle(f"Labelled: {posture}", fontsize=16)

    for axis, (_, row) in zip(axes.flat, picked_rows.iterrows()):
        sample = load_sample("posture", row)
        axis.imshow(sample.image)
        
        # Draw bounding boxes for each image
        x, y, w, h = sample.box
        axis.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", linewidth=2))

        # Add label and clip_id as title
        axis.set_title(row["clip_id"].split("/")[-1], fontsize=8)
        axis.axis("off")
    
    plt.tight_layout()
    plt.show()

show_posture("lying")
show_posture("standing")